# 5.2 — Configuration and Precedence

**Chapter 5, sections 5.3 and 5.4**, and the starting point for **Exercise 3**.

**The question this notebook answers:** the chapter says practitioners lose a great deal of time
to properties that were set correctly and applied to nothing. That is a claim about a mechanism,
and the only way to show *silently ignored* is to set a property, then read back what the
application actually received and find it unchanged — with no exception raised anywhere.

Three distinctions are separated here, and they are usually confused with one another:

1. **Precedence** — three places may set a property, and one of them wins.
2. **Launch time against run time** — some properties are read when a process is created and
   can never be changed afterwards, whatever the precedence says.
3. **Who creates the session** — and this turns out to decide the behaviour of the chapter's
   own headline example, in a way the chapter does not yet record.

Runs on a laptop in about a minute, most of it spent submitting four small applications.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, sys, json, time, tempfile, logging, subprocess, urllib.request
from urllib.parse import urlparse
import pandas as pd
import pyspark
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

# Two properties are set on the builder, deliberately: one runtime-modifiable, one launch-time.
spark = (SparkSession.builder
         .appName("CS777-5.2")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", 240)     # runtime-modifiable
         .config("spark.driver.memory", "2g")             # launch-time
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
for _lg in ("SQLQueryContextLogger", "DataFrameQueryContextLogger"):
    logging.getLogger(_lg).setLevel(logging.CRITICAL)

pd.set_option("display.width", 200)

_port = urlparse(sc.uiWebUrl).port
UI = f"http://localhost:{_port}/api/v1"
APP = json.load(urllib.request.urlopen(f"{UI}/applications"))[0]["id"]

def ui(path):
    with urllib.request.urlopen(f"{UI}/applications/{APP}{path}", timeout=60) as r:
        return json.load(r)

print("Spark", spark.version, "| session created by: this program")

Spark 4.2.0 | session created by: this program


## 1. The Environment tab

The chapter's advice whenever a setting appears to have had no effect is to stop reading the
source and look at the Environment tab, which lists the properties the application actually
received. That page is an endpoint like any other.

In [2]:
env = ui("/environment")
props = dict(env["sparkProperties"])

INTERESTING = ["spark.app.name", "spark.master", "spark.submit.deployMode",
               "spark.driver.memory", "spark.executor.memory", "spark.executor.cores",
               "spark.sql.shuffle.partitions", "spark.sql.files.maxPartitionBytes",
               "spark.dynamicAllocation.enabled"]

rows = [{"property": k,
         "in the Environment tab": props.get(k, "<absent: the default applies>"),
         "modifiable": spark.conf.isModifiable(k)}
        for k in INTERESTING]
print(pd.DataFrame(rows).to_string(index=False))
print("\nruntime:", env["runtime"])
print(f"\n{len(props)} properties listed in all; the ones not listed are running on defaults.")

                         property        in the Environment tab  modifiable
                   spark.app.name                     CS777-5.2       False
                     spark.master                      local[*]       False
          spark.submit.deployMode                        client       False
              spark.driver.memory                            2g       False
            spark.executor.memory <absent: the default applies>       False
             spark.executor.cores <absent: the default applies>       False
     spark.sql.shuffle.partitions                           240        True
spark.sql.files.maxPartitionBytes <absent: the default applies>        True
  spark.dynamicAllocation.enabled <absent: the default applies>       False

runtime: {'javaVersion': '21.0.12.1 (Eclipse Adoptium)', 'javaHome': '/Library/Java/JavaVirtualMachines/temurin-21.jdk/Contents/Home', 'scalaVersion': 'version 2.13.18'}

22 properties listed in all; the ones not listed are running on defa

## 2. Launch-time and runtime-modifiable

`isModifiable` in the table above is the chapter's second distinction, and in practice it
matters more than precedence. A runtime-modifiable property is consulted every time a query is
planned; a launch-time property was read when the process was created and nothing can change it
now.

The two kinds fail differently, and the difference is the point of this section: **one raises and
one is silent.**

In [3]:
# The modifiable one: changed on a live session, and it takes effect from the next shuffle.
before = spark.conf.get("spark.sql.shuffle.partitions")
spark.conf.set("spark.sql.shuffle.partitions", 480)
print(f"spark.conf.set on a modifiable property : {before} -> "
      f"{spark.conf.get('spark.sql.shuffle.partitions')}   (no complaint, and it is real)")
spark.conf.set("spark.sql.shuffle.partitions", before)

# The launch-time one, route A: spark.conf.set. This path refuses.
try:
    spark.conf.set("spark.executor.memory", "16g")
    print("spark.conf.set on a launch-time property : accepted -- unexpected")
except Exception as e:
    first = str(e).split("\n")[0]
    print(f"spark.conf.set on a launch-time property : {type(e).__name__}")
    print(f"                                          {first[:96]}")

spark.conf.set on a modifiable property : 240 -> 480   (no complaint, and it is real)
spark.conf.set on a launch-time property : AnalysisException
                                          [CANNOT_MODIFY_CONFIG] Cannot modify the value of the Spark config: "spark.executor.memory".


### Route B: the builder, on a session that already exists

`SparkSession.builder` does not create a second session when one is already running. It returns
the one that exists and hands it the new options, and what happens to them depends on the kind of
property. Nothing is raised either way.

Three properties are set below on a live session: one runtime-modifiable, one launch-time, and
the application's own name. Each is then read back from **three** places, because they do not
agree, and which of the three you happen to consult decides whether you notice.

* `spark.conf.get` — the *session's* view, including overrides stored on the session.
* the **Environment tab** — what the application was launched with.
* `sc.getConf()` — the `SparkConf` the `SparkContext` was built from.

In [4]:
def three_views(key):
    try:
        session_view = spark.conf.get(key)
    except Exception:
        session_view = "<raises>"
    return {"property": key,
            "spark.conf.get": session_view,
            "Environment tab": dict(ui("/environment")["sparkProperties"]).get(key, "<absent>"),
            "sc.getConf()": sc.getConf().get(key, "<absent>")}

KEYS = ["spark.sql.shuffle.partitions", "spark.executor.memory", "spark.app.name"]
print("before:")
print(pd.DataFrame([three_views(k) for k in KEYS]).to_string(index=False))

app_name_before = json.load(urllib.request.urlopen(f"{UI}/applications"))[0]["name"]

same = (SparkSession.builder
        .appName("a completely different name")
        .config("spark.executor.memory", "16g")
        .config("spark.sql.shuffle.partitions", 999)
        .getOrCreate())

print(f"\na new session object?  {same is not spark}      (the builder returned the one that existed)\n")
print("after:")
print(pd.DataFrame([three_views(k) for k in KEYS]).to_string(index=False))

app_name_after = json.load(urllib.request.urlopen(f"{UI}/applications"))[0]["name"]
print(f"\nthe application's name in the Jobs tab: {app_name_before!r} -> {app_name_after!r}")

before:
                    property spark.conf.get Environment tab sc.getConf()
spark.sql.shuffle.partitions            240             240          240
       spark.executor.memory       <raises>        <absent>     <absent>
              spark.app.name      CS777-5.2       CS777-5.2    CS777-5.2



a new session object?  False      (the builder returned the one that existed)

after:


                    property              spark.conf.get Environment tab sc.getConf()
spark.sql.shuffle.partitions                         999             240          240
       spark.executor.memory                         16g        <absent>     <absent>
              spark.app.name a completely different name       CS777-5.2    CS777-5.2

the application's name in the Jobs tab: 'CS777-5.2' -> 'CS777-5.2'


Read the second table one row at a time, because the three rows did three different things.

**`spark.sql.shuffle.partitions` really changed.** It is runtime-modifiable, so the builder
applied it, and 999 is what the next shuffle will use. This is the case that works.

**`spark.executor.memory` did not change — and `spark.conf.get` says it did.** The executors were
launched with the old value and are still running with it, which is what the Environment tab and
`sc.getConf()` report. What the builder did was store a session-local override that only
`spark.conf.get` can see. So this is worse than the chapter's "silently ignored": the property is
silently ignored *and* the most convenient way of checking reports success.

**`spark.app.name` is the same trick in its purest form.** The session's conf now holds the new
name; the application in the Jobs tab is still called what it was called when it started.

The rule that survives all three rows: **`spark.conf.get` reports what this session was told, and
the Environment tab reports what the application was launched with.** When a launch-time property
is in question, the second is the one that is load-bearing — which is exactly why the chapter
sends you there. The next section is the one case where even that is not enough.

## 3. The driver's memory, which behaves differently than the chapter says

§5.3 gives `spark.driver.memory` as the sharpest instance of a launch-time property: *"in client
mode the driver JVM already exists by the time the program's first line runs, so
`spark.driver.memory` set on the session has nothing left to configure."*

That is true of an application submitted with `spark-submit`, and it is **not** true of this
notebook. The distinction is who starts the JVM. Under `spark-submit`, the launcher starts the
driver JVM and then hands control to the program, so the heap is already fixed. In a notebook or
a plain `python script.py`, no JVM exists until `getOrCreate()` is called, and PySpark starts one
*then* — after it has read the builder's configuration, which it passes straight through as the
heap size.

Both cases are run below. The property is asked for in exactly the same line of code in both;
what differs is the launcher.

In [5]:
SPARK_HOME = os.path.dirname(pyspark.__file__)
SUBMIT = os.path.join(SPARK_HOME, "bin", "spark-submit")
SCRIPT = os.path.join("..", "scripts", "05.02 Effective Config.py")

def submit(*args, script_args=()):
    """Run the shipped script under spark-submit and parse its CS777-CONFIG lines."""
    env = dict(os.environ, SPARK_HOME=SPARK_HOME,
               PYSPARK_PYTHON=sys.executable, PYSPARK_DRIVER_PYTHON=sys.executable)
    out = subprocess.run([SUBMIT, "--master", "local[2]", *args, SCRIPT, *script_args],
                         capture_output=True, text=True, env=env, timeout=300)
    conf = {}
    for line in out.stdout.splitlines():
        if line.startswith("CS777-CONFIG "):
            body = line[len("CS777-CONFIG "):]
            key, _, rest = body.partition("=")
            conf[key] = rest.rsplit(" modifiable=", 1)[0]
    if not conf:
        raise RuntimeError(out.stderr[-2000:])
    return conf

def run_as_plain_python():
    """The same script, started by Python rather than by spark-submit."""
    env = dict(os.environ, PYSPARK_PYTHON=sys.executable)
    out = subprocess.run([sys.executable, SCRIPT], capture_output=True, text=True,
                         env=env, timeout=300)
    return {line[len("CS777-CONFIG "):].partition("=")[0]:
            line[len("CS777-CONFIG "):].partition("=")[2].rsplit(" modifiable=", 1)[0]
            for line in out.stdout.splitlines() if line.startswith("CS777-CONFIG ")}

print("The script asks for spark.driver.memory = 3g on its session builder, and nothing else.\n")
t0 = time.time()
by_python = run_as_plain_python()
by_submit = submit()
by_submit_flag = submit("--driver-memory", "2g")
print(f"(three applications submitted in {time.time() - t0:.0f} s)\n")

table = pd.DataFrame([
    {"launcher": "python script.py",
     "extra option": "-",
     "property reports": by_python["spark.driver.memory"],
     "JVM heap MiB": by_python["driver.jvm.maxHeapMiB"]},
    {"launcher": "spark-submit",
     "extra option": "-",
     "property reports": by_submit["spark.driver.memory"],
     "JVM heap MiB": by_submit["driver.jvm.maxHeapMiB"]},
    {"launcher": "spark-submit",
     "extra option": "--driver-memory 2g",
     "property reports": by_submit_flag["spark.driver.memory"],
     "JVM heap MiB": by_submit_flag["driver.jvm.maxHeapMiB"]},
])
print(table.to_string(index=False))

The script asks for spark.driver.memory = 3g on its session builder, and nothing else.



(three applications submitted in 8 s)

        launcher       extra option property reports JVM heap MiB
python script.py                  -               3g         3072
    spark-submit                  -               3g         1024
    spark-submit --driver-memory 2g               3g         2048


### Two conclusions, and the second one is a caution about the chapter's own advice

**The builder line works when the program starts the JVM and is ignored when the launcher does.**
Row one got the heap it asked for; row two asked for exactly the same thing and got the 1 GiB
default. Row three shows the documented remedy working, and shows something else besides: the
property still *reports* 3 g, because the program did set it, while the heap is the 2 GiB the
flag asked for. The reported value and the effective value are two different things.

**So the Environment tab does not settle this particular question.** §5.3 says such a question is
settled there rather than in the source code, and for a runtime-modifiable property that is
exactly right. For `spark.driver.memory` it is not: the page reports what was *requested*, and
all three rows above would show a plausible value there. The only witness that cannot be argued
with is the JVM's own heap, which is what the last column reads.

```python
sc._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**2   # what the driver actually has
```

In [6]:
print("this notebook's own driver:")
print("   property reports :", spark.conf.get("spark.driver.memory"))
print("   Environment tab  :", dict(ui("/environment")["sparkProperties"])
                               .get("spark.driver.memory", "<absent>"))
print("   JVM actually has :",
      round(sc._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1048576), "MiB")
print()
print("All three agree here, because this notebook started its own JVM.")
print("Under spark-submit the third line is the one that would have disagreed.")

this notebook's own driver:
   property reports : 2g
   Environment tab  : 2g
   JVM actually has : 2048 MiB

All three agree here, because this notebook started its own JVM.
Under spark-submit the third line is the one that would have disagreed.


## 4. Precedence, which needs more than one application to demonstrate

Three places may set a property, in increasing order of precedence: the defaults file, the
submission command, and the program. A single session cannot show this, because it only ever
sees the winner. The shipped script sets `spark.sql.files.maxPartitionBytes` to 64 m in the
program, and below it is submitted against a defaults file and a `--conf` flag that ask for
something else.

In [7]:
DEFAULTS = os.path.join(SCRATCH, "cs777-defaults.conf")
with open(DEFAULTS, "w") as f:
    f.write("spark.sql.files.maxPartitionBytes 16m\n")
    f.write("spark.sql.shuffle.partitions      111\n")

runs = [
    ("nothing but the program",           submit()),
    ("--properties-file (defaults file)",  submit("--properties-file", DEFAULTS)),
    ("--conf on the command line",         submit("--conf", "spark.sql.files.maxPartitionBytes=32m",
                                                  "--conf", "spark.sql.shuffle.partitions=222")),
    ("defaults file AND --conf",           submit("--properties-file", DEFAULTS,
                                                  "--conf", "spark.sql.files.maxPartitionBytes=32m",
                                                  "--conf", "spark.sql.shuffle.partitions=222")),
]

prec = pd.DataFrame([
    {"submitted with": label,
     "maxPartitionBytes": conf["spark.sql.files.maxPartitionBytes"],
     "shuffle.partitions": conf["spark.sql.shuffle.partitions"]}
    for label, conf in runs])
print("the program asks for maxPartitionBytes=64m and says nothing about shuffle.partitions\n")
print(prec.to_string(index=False))

the program asks for maxPartitionBytes=64m and says nothing about shuffle.partitions

                   submitted with maxPartitionBytes shuffle.partitions
          nothing but the program               64m                200
--properties-file (defaults file)               64m                111
       --conf on the command line               64m                222
         defaults file AND --conf               64m                222


Read the two columns differently, because they are showing two different things.

**`maxPartitionBytes`** is set in the program in every row, and the program wins in every row —
64 m against the defaults file's 16 m and against `--conf`'s 32 m. That is the precedence order,
demonstrated.

**`shuffle.partitions`** is *not* set in the program, so it shows what the lower two places do
when the highest is silent: the defaults file supplies 111, `--conf` supplies 222, and when both
are present `--conf` wins. A property left alone by the program is the only way to see the
bottom two rungs of the ladder at all.

The practical rule the chapter draws from this is about *where* rather than *whether*: set a
property as late as it is permitted to be set and no later. Runtime-modifiable properties belong
in the program, where they can be tuned per query; launch-time properties belong in the
submission command, beside the job they govern; and the defaults file is for settings that
genuinely apply to every job on the installation.

## 5. The submission flags are aliases

§5.4 notes that the sizing flags are not a separate mechanism: `--executor-memory` is
`spark.executor.memory`, `--executor-cores` is `spark.executor.cores`, `--num-executors` is
`spark.executor.instances`. Anything without a dedicated flag goes through `--conf`.

In [8]:
by_flag = submit("--executor-memory", "3g", "--executor-cores", "2")
by_conf = submit("--conf", "spark.executor.memory=3g", "--conf", "spark.executor.cores=2")

# spark.driver.memory is left out of this table on purpose: the script sets it in the program,
# so what it reports says nothing about the flag. Section 3 above is where that one is settled.
print(pd.DataFrame([
    {"property": "spark.executor.memory",
     "via the flag": by_flag["spark.executor.memory"],
     "via --conf": by_conf["spark.executor.memory"]},
    {"property": "spark.executor.cores",
     "via the flag": by_flag["spark.executor.cores"],
     "via --conf": by_conf["spark.executor.cores"]},
]).to_string(index=False))

print()
print("deploy mode reported by these runs:", by_flag["spark.submit.deployMode"])

             property via the flag via --conf
spark.executor.memory    <not set>         3g
 spark.executor.cores    <not set>          2

deploy mode reported by these runs: client


The executor flags did nothing, and the reason is `--master local[2]`.

`spark-submit` assigns `--executor-memory` and `--executor-cores` to their properties only for
the cluster managers — standalone, YARN, Kubernetes. In local mode there are no executor
processes to size: the work runs in threads inside the driver JVM, so the flags are dropped and
the corresponding properties are never set. Passing the same values through `--conf` sets the
properties regardless, because `--conf` is a general mechanism with no such filter; the values
then sit in the configuration doing nothing, which is its own kind of misleading.

The lesson is the chapter's own, arrived at from a new direction: **the flag you typed is not
evidence that the property was set.** The Environment tab is, and it is as informative about a
property that is *absent* as about one that is present.

Client mode is the default, which is why this cell could print anything at all: the driver ran
inside the `spark-submit` process and its standard output came back here. In cluster mode the
driver would be on a worker node and this output would be in the cluster manager's logs.

### A note for anyone running this from a path with a space in it

The course tree lives under `My Drive`, and PySpark's `spark-submit` shell wrapper does not quote
the path it derives for `SPARK_HOME`. Run it directly and it fails with
`/Users/…/My: No such file or directory` before Spark starts at all. Setting `SPARK_HOME`
explicitly skips the offending step, which is what the `submit` helper above does:

```bash
export SPARK_HOME="$(python -c 'import pyspark,os; print(os.path.dirname(pyspark.__file__))')"
"$SPARK_HOME/bin/spark-submit" --master 'local[2]' "05.02 Effective Config.py"
```

This is an environment defect rather than a Spark one, but it costs an hour the first time and
it has nothing to do with the configuration being studied.

## Conclusion

A property has three places it may be set and two moments at which it may be read, and the two
questions are independent. Precedence decides which of three values is used; launch time against
run time decides whether that value can still do anything when it arrives.

What this notebook establishes by running it:

1. **A launch-time property fails in two different ways.** Through `spark.conf.set` it raises
   `CANNOT_MODIFY_CONFIG`. Through the builder on a session that already exists it is discarded
   without a word, and that silence is the reason this is a section of the chapter rather than a
   footnote.
2. **Who starts the JVM decides whether `spark.driver.memory` on the builder works.** In this
   notebook it works, because PySpark starts the driver JVM at `getOrCreate()` and passes the
   value through. Under `spark-submit` the same line does nothing, because the JVM is older than
   the program.
3. **The Environment tab reports what was requested.** For `spark.driver.memory` that is not the
   same as what the driver got, and the JVM's own `maxMemory` is the only witness that settles
   it.
4. **Precedence runs program > `--conf` > defaults file**, shown on a property the program sets
   and again on one it leaves alone.

*Chapter sections:* §5.3 (setting Spark configuration), §5.4 (`spark-submit` options).
*Exercise 3* asks where each of four properties must be set and what happens if it is set on the
session in client mode; sections 2 and 3 above answer both halves, and the `isModifiable` column
of section 1 is how to check any property not on that list.